In [27]:
from math import factorial
debugging_mode=True
from pyspark.sql.functions import (
    col, year, month, dayofmonth, weekofyear, date_format,
    weekday, when, expr, to_date, row_number
)
from pyspark.sql.window import Window
from datetime import datetime
import ConnectionConfig as cc
from pyspark.sql.types import StringType
from pyspark.sql.functions import udf
from pyspark.sql.functions import col, to_date, first

cc.setupEnvironment()


Environment variables are set...


In [2]:
#config
cc.setupEnvironment()
print(cc.config.sections())

Environment variables are set...
['default', 'tutorial_op', 'catchem', 'kafka']


In [3]:
spark = cc.startLocalCluster("FACT_TREASURE_FOUND",4)
spark.getActiveSession()

In [4]:

#make connection
cc.config.read('config.ini')
cc.set_connectionProfile("catchem")
run_timestamp = datetime.now()


In [5]:
#zet al de dim availeble als views
dateDimDf= spark.read.format("delta").load("./delta/DATE_DIM")
rainDimDf= spark.read.format("delta").load("./delta/RAIN_DIM")
seasonDimDf= spark.read.format("delta").load("./delta/SEASON_DIM")
userDimDf = spark.read.format("delta").load("./spark-warehouse/dimuser")
treasureTypeDimDf = spark.read.format("delta").load("./spark-warehouse/dimtreasuretype")

dateDimDf.createOrReplaceTempView("dimDate")
rainDimDf.createOrReplaceTempView("dimeRain")
seasonDimDf.createOrReplaceTempView("dimSeason")
userDimDf.createOrReplaceTempView("dimUser")
treasureTypeDimDf.createOrReplaceTempView("dimTreasureType")

In [6]:
#select de treasure logs
df_users = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "treasure_log") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .load()

df_users.createOrReplaceTempView("treasure_log")

df_users = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "treasure") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .load()

df_users.createOrReplaceTempView("treasure")

df_users = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "treasure_stages") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .load()

df_users.createOrReplaceTempView("treasure_stages")

df_users = spark.read \
    .format("jdbc") \
    .option("driver", cc.get_Property("driver")) \
    .option("url", cc.create_jdbc()) \
    .option("dbtable", "stage") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .load()

df_users.createOrReplaceTempView("stages")

In [46]:
#maak de feit tabel

treasureFoundFact = spark.sql(f"""
SELECT
    CAST((unix_timestamp(tl.log_time) - unix_timestamp(tl.session_start)) AS BIGINT) as duration,
    1 AS default_value,
    to_timestamp('{run_timestamp}') as CreationDate,
    to_date(tl.log_time) as LogDate,
    dd.DateSurKey as DateSurKey,
    us.UserSurKey as UserSurKey,
    tt.TreasureTypeSurKey as TreasureTypeSurKey,
    month(tl.log_time) as MonthOfTheYear,
    s.latitude as latitude,
    t.id as treasure_id
FROM treasure_log tl
LEFT OUTER JOIN dimDate as dd ON (
    year(tl.log_time) = dd.Year
    AND month(tl.log_time) = dd.MonthOfTheYear
    AND day(tl.log_time) = dd.Day
)
LEFT OUTER JOIN dimUser as us on tl.hunter_id = us.userId
LEFT OUTER JOIN treasure as t ON tl.treasure_id = t.id
LEFT OUTER JOIN (
    SELECT treasure_id, COUNT(stages_id) as number_of_stages
    FROM treasure_stages GROUP BY treasure_id
) ts ON tl.treasure_id = ts.treasure_id
LEFT OUTER JOIN dimTreasureType as tt ON (
    t.difficulty = tt.Difficulty AND t.terrain = tt.Terrain AND ts.number_of_stages = tt.Size
)
LEFT OUTER JOIN (
    SELECT
        ts.treasure_id,
        AVG(s.latitude) as latitude
    FROM treasure_stages ts
    LEFT OUTER JOIN stages s ON ts.stages_id = s.id
    GROUP BY ts.treasure_id
) s ON tl.treasure_id = s.treasure_id
WHERE tl.log_type = 2
""")

treasureFoundFact.createOrReplaceTempView("treasureFoundFact")
spark.sql("SELECT * FROM treasureFoundFact LIMIT 10").show()


+--------+-------------+--------------------+----------+--------------------+----------+------------------+--------------+------------------+--------------------+
|duration|default_value|        CreationDate|   LogDate|          DateSurKey|UserSurKey|TreasureTypeSurKey|MonthOfTheYear|          latitude|         treasure_id|
+--------+-------------+--------------------+----------+--------------------+----------+------------------+--------------+------------------+--------------------+
|    1080|            1|2025-11-02 00:14:...|2023-06-29|933fa58e-f359-4a2...|    377302|               236|             6| 26.54750671890924|[FE F6 FD 12 2C A...|
|      60|            1|2025-11-02 00:14:...|2022-03-29|6d1ba0f2-a2f7-47f...|     69178|               172|             3| 53.56129911229332|[3D C0 F9 49 D0 1...|
|    2700|            1|2025-11-02 00:14:...|2021-04-14|97764095-9e10-47d...|    396110|               104|             4| 23.40785437806007|[4F 7A EC 20 00 8...|
|    8400|            

In [36]:
# 2. SeasonDim koppelen
def get_season(month_of_year, latitude):
    if latitude is None or month_of_year is None: return "UNKNOWN"
    if latitude > 0: # Noordelijk halfrond
        if 3 <= month_of_year <= 5: return "Lente"
        elif 6 <= month_of_year <= 8: return "Zomer"
        elif 9 <= month_of_year <= 11: return "Herfst"
        else: return "Winter"
    elif latitude < 0: # Zuidelijk halfrond
        if month_of_year >= 9 and month_of_year <= 11: return "Lente"
        elif month_of_year >= 12 or month_of_year <= 2: return "Zomer"
        elif month_of_year >= 3 and month_of_year <= 5: return "Herfst"
        else: return "Winter"
    return None


In [37]:
get_season_udf = udf(get_season, StringType())

fact_with_season_and_id = treasureFoundFact.withColumn(
    "DeterminedSeason",
    get_season_udf(col("MonthOfTheYear"), col("latitude"))
).join(
    seasonDimDf.select("SeasonName", "SeasonSurKey"),
    col("DeterminedSeason") == col("SeasonName"),
    how="left"
).drop("DeterminedSeason", "SeasonName", "MonthOfTheYear", "latitude")

print("fact_with_season_key preview:")
fact_with_season_and_id.show(5)

fact_with_season_key preview:
+--------+-------------+--------------------+----------+--------------------+----------+------------------+--------------------+--------------------+
|duration|default_value|        CreationDate|   LogDate|          DateSurKey|UserSurKey|TreasureTypeSurKey|         treasure_id|        SeasonSurKey|
+--------+-------------+--------------------+----------+--------------------+----------+------------------+--------------------+--------------------+
|    1080|            1|2025-11-02 00:14:...|2023-06-29|933fa58e-f359-4a2...|    377302|               236|[FE F6 FD 12 2C A...|a9da85a1-9f12-41e...|
|      60|            1|2025-11-02 00:14:...|2022-03-29|6d1ba0f2-a2f7-47f...|     69178|               172|[3D C0 F9 49 D0 1...|66c0a3df-5cb6-4a4...|
|    8400|            1|2025-11-02 00:14:...|2021-08-04|1b1c941c-e839-4c9...|    385460|               149|[AB CD D7 35 CE 2...|a9da85a1-9f12-41e...|
|    2400|            1|2025-11-02 00:14:...|2023-06-06|4bbdfbb2-f664-

In [38]:
city_src = (
    spark.read
    .format("jdbc")
    .option("url", cc.create_jdbc())
    .option("driver", cc.get_Property("driver"))
    .option(
        "dbtable",
        "(select city_id, postal_code from city) as subq"
    )
    .option("user", cc.get_Property("username"))
    .option("password", cc.get_Property("password"))
    .load()
)
print("city_src preview:")
city_src.show(15)

city_src preview:
+--------------------+-----------+
|             city_id|postal_code|
+--------------------+-----------+
|[00 00 0B 50 FA 3...|   513-1121|
|[00 00 3E 94 C1 D...|     678 01|
|[00 00 44 4E FB C...|       3133|
|[00 00 46 73 9F A...|      35320|
|[00 00 4A B0 6E 4...|   500-8844|
|[00 00 60 7D 16 1...|     152636|
|[00 00 7E 93 82 4...|     413528|
|[00 00 9A 97 B5 9...|      66072|
|[00 00 A2 6E C3 C...|   465-0058|
|[00 00 BC 55 B4 B...|      48200|
|[00 00 C3 10 D2 3...|      07046|
|[00 00 C6 BB 5E B...|   2690-563|
|[00 00 D8 F6 1D D...|      78570|
|[00 00 FA C7 77 E...|      27958|
|[00 01 40 AF 63 3...|        HS7|
+--------------------+-----------+
only showing top 15 rows


In [39]:
weather_history_df = spark.read.option("multiline", "true").json("./Weatherhistory/weerhistoriek.json")
weather_history_df.show(30, truncate=False)

+--------------------+----------------+--------------------+------------------------------------------------+----------+--------+
|city                |main            |timestamp           |weather                                         |wind      |zipCode |
+--------------------+----------------+--------------------+------------------------------------------------+----------+--------+
|Kishitacho          |{22.1, 55, 22.5}|2025-10-05T08:00:00Z|{clear sky, 01d, 800, Clear}                    |{150, 3.2}|513-1121|
|Kishitacho          |{25.0, 60, 24.8}|2025-10-05T12:00:00Z|{few clouds, 02d, 801, Clouds}                  |{190, 4.1}|513-1121|
|Kishitacho          |{20.0, 75, 20.2}|2025-10-05T16:00:00Z|{light rain, 10d, 500, Rain}                    |{220, 5.0}|513-1121|
|Blansko             |{13.2, 92, 13.5}|2025-10-05T07:00:00Z|{mist, 50d, 741, Mist}                          |{90, 1.8} |678 01  |
|Blansko             |{18.1, 70, 18.4}|2025-10-05T13:00:00Z|{broken clouds, 04d, 803, Clou

In [47]:
from pyspark.sql.functions import to_timestamp, to_date, col

weather_flat = weather_history_df.select(
    col("zipCode"),
    to_date(to_timestamp(col("timestamp"), "yyyy-MM-dd'T'HH:mm:ssX")).alias("weather_date"),
    col("weather.id").alias("weather_id")
)


In [48]:

# === KOPPEL WEERDATA AAN STAD + DATUM ===
# Eerst city_src (stad/postcode)
print("city_src preview:")
city_src.show(15)

# Alle steden behouden, ook zonder weerdata
city_with_weather = city_src.join(
    weather_flat,
    city_src["postal_code"] == weather_flat["zipCode"],
    "left"
)


print("Resultaat join stad + weer:")
city_with_weather.show(40, truncate=False)


city_src preview:
+--------------------+-----------+
|             city_id|postal_code|
+--------------------+-----------+
|[00 00 0B 50 FA 3...|   513-1121|
|[00 00 3E 94 C1 D...|     678 01|
|[00 00 44 4E FB C...|       3133|
|[00 00 46 73 9F A...|      35320|
|[00 00 4A B0 6E 4...|   500-8844|
|[00 00 60 7D 16 1...|     152636|
|[00 00 7E 93 82 4...|     413528|
|[00 00 9A 97 B5 9...|      66072|
|[00 00 A2 6E C3 C...|   465-0058|
|[00 00 BC 55 B4 B...|      48200|
|[00 00 C3 10 D2 3...|      07046|
|[00 00 C6 BB 5E B...|   2690-563|
|[00 00 D8 F6 1D D...|      78570|
|[00 00 FA C7 77 E...|      27958|
|[00 01 40 AF 63 3...|        HS7|
+--------------------+-----------+
only showing top 15 rows
Resultaat join stad + weer:
+-------------------------------------------------+-----------+--------+------------+----------+
|city_id                                          |postal_code|zipCode |weather_date|weather_id|
+-------------------------------------------------+-----------+-------

In [49]:

# === Voeg RainSurKey toe ===
from pyspark.sql.functions import when

city_with_rain_key = city_with_weather.withColumn(
    "RainSurKey",
    when((col("weather_id") >= 200) & (col("weather_id") <= 699), 1)  # met regen
    .when(col("weather_id").isNotNull(), 2)                           # zonder regen
    .otherwise(3)                                                     # onbekend
)


city_with_rain_key.select("city_id", "postal_code", "weather_id", "weather_date", "RainSurKey").show(40)


+--------------------+-----------+----------+------------+----------+
|             city_id|postal_code|weather_id|weather_date|RainSurKey|
+--------------------+-----------+----------+------------+----------+
|[00 00 0B 50 FA 3...|   513-1121|       500|  2025-10-05|         1|
|[00 00 0B 50 FA 3...|   513-1121|       801|  2025-10-05|         2|
|[00 00 0B 50 FA 3...|   513-1121|       800|  2025-10-05|         2|
|[00 00 3E 94 C1 D...|     678 01|       800|  2025-10-05|         2|
|[00 00 3E 94 C1 D...|     678 01|       803|  2025-10-05|         2|
|[00 00 3E 94 C1 D...|     678 01|       741|  2025-10-05|         2|
|[00 00 44 4E FB C...|       3133|       802|  2025-10-05|         2|
|[00 00 44 4E FB C...|       3133|       211|  2025-10-05|         1|
|[00 00 44 4E FB C...|       3133|       500|  2025-10-05|         1|
|[00 00 46 73 9F A...|      35320|       500|  2025-10-05|         1|
|[00 00 46 73 9F A...|      35320|       803|  2025-10-05|         2|
|[00 00 46 73 9F A..

In [50]:

# === Treasure inladen ===
treasure_src = (
    spark.read
        .format("jdbc")
        .option("url", cc.create_jdbc())
        .option("driver", cc.get_Property("driver"))
        .option(
            "dbtable",
            "(select id, city_city_id from treasure) as subq"
        )
        .option("user", cc.get_Property("username"))
        .option("password", cc.get_Property("password"))
        .load()
)
print("treasure_src preview:")
treasure_src.show(15)


treasure_src preview:
+--------------------+--------------------+
|                  id|        city_city_id|
+--------------------+--------------------+
|[00 00 3E 2C B1 4...|[62 34 F0 0E 0E 9...|
|[00 03 72 3C 7C C...|[59 07 42 E8 55 1...|
|[00 04 39 A0 98 7...|[DB 39 44 FB 1D 0...|
|[00 04 62 1B 29 E...|[66 97 B3 4B 1F 1...|
|[00 05 1E A2 05 8...|[38 EC 6D 3C 5A 1...|
|[00 05 A4 FF 38 0...|[99 F1 55 1E 80 8...|
|[00 06 05 10 FC 0...|[50 3E 91 AA F3 3...|
|[00 08 A9 BF BE 4...|[59 B1 A5 28 DD C...|
|[00 08 B8 9F B1 D...|[F6 A7 AD 78 3F 1...|
|[00 08 B9 20 1B F...|[27 9E 9E 63 9E A...|
|[00 08 E2 70 CB A...|[AE A0 A3 09 22 1...|
|[00 09 57 54 16 D...|[B2 BE 60 8E A5 B...|
|[00 09 5D DB 3B C...|[79 FC E5 64 B9 B...|
|[00 0A 1E 88 C1 1...|[2F 08 AB 1D 02 B...|
|[00 0A 91 0B 0E 3...|[63 58 42 A3 F6 5...|
+--------------------+--------------------+
only showing top 15 rows


In [53]:
from pyspark.sql.functions import coalesce, lit

fact_complete = (
    fact_with_season_and_id
    .join(treasure_src, fact_with_season_and_id["treasure_id"] == treasure_src["id"], "left")
    .join(
        city_with_rain_key.select("city_id", "weather_date", "RainSurKey"),
        (treasure_src["city_city_id"] == city_with_rain_key["city_id"]) &
        (fact_with_season_and_id["LogDate"] == city_with_rain_key["weather_date"]),
        "left"
    )
    .withColumn("RainSurKey", coalesce(col("RainSurKey"), lit(3)))  # vul null met 3
    .drop("id", "city_city_id", "city_id", "treasure_id", "weather_date")
)
fact_complete.show(20, truncate=False)
fact_complete.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save("./delta/FACT_TREASURE_FOUND")


+--------+-------------+--------------------------+----------+------------------------------------+----------+------------------+------------------------------------+----------+
|duration|default_value|CreationDate              |LogDate   |DateSurKey                          |UserSurKey|TreasureTypeSurKey|SeasonSurKey                        |RainSurKey|
+--------+-------------+--------------------------+----------+------------------------------------+----------+------------------+------------------------------------+----------+
|4200    |1            |2025-11-02 00:14:01.831969|2022-09-02|2fa63796-cb27-4471-b2e9-fd3fd81c965f|93004     |135               |928f6f64-6761-4728-a10a-63150f9782f9|3         |
|6060    |1            |2025-11-02 00:14:01.831969|2022-01-08|af888b4f-41c8-47a0-bd6b-21db4773bda1|364728    |130               |4e770979-e9fb-4f27-9c78-b227d8c4805e|3         |
|3720    |1            |2025-11-02 00:14:01.831969|2021-08-18|cd08f4f6-7fd4-48bb-93ce-7992573cbcc8|244864    |